# 모델 B: 감성 점수 대신 임베딩 표현으로 ε를 예측할 수 있는가
**질문**: 같은 텍스트·같은 ε·같은 평가 틀에서, FinBERT 감성 점수(1차원 요약) 대신 문장 임베딩 기반 표현을 쓰면 A 단독보다 나아지는가?

- 라벨 ε = real_ret − pred_ret(Chronos-2, H=3). 최종 평가는 r̂ = A + ε̂ 와 A 단독의 날짜별 횡단면 IC 비교.
- 평가 틀은 PR #16(`experiment.py`)과 동일: walk-forward 5블록(첫 40% 학습 전용), 학습은 테스트 시작 3거래일 전까지(purge), t값은 독립 표본을 n/3으로 줄여 계산.
- 추가: ΔIC(최종 − A 단독)에 **블록 부트스트랩 95% 구간**(블록 5일)을 붙인다.
- 인코더: `all-MiniLM-L6-v2` (뉴스·Reddit 공통).

| 방법 | 학습 | 설명 |
|---|---|---|
| FinBERT | 계수 1개 | 기존 감성 점수 일평균 × β (기준선) |
| 정의 유사도 ×3 | 계수 1개 | 호재/악재 정의문과의 코사인 유사도 차 (정의문 3종) |
| ε-centroid | fold 학습 | 학습 구간에서 ε>0 / ε<0 인 날의 평균 임베딩과의 유사도 차 |
| 임베딩 + Ridge / MLP | fold 학습 | 일평균 임베딩 → ε. 하이퍼파라미터는 학습 구간 마지막 20%로 선택, 검증 구간에서 크기 보정(β≥0) |

**준비:** 런타임 GPU. `RANGE`를 `'6m'`(2026-05~09) 또는 `'full'`(2021-01~2026-09)로 정하고 `news_{RANGE}.parquet`, `reddit_{RANGE}.parquet`, `modelA_{RANGE}.csv` 업로드. Reddit 원문은 공개 금지.

In [ ]:
%pip -q install sentence-transformers   # sklearn/scipy는 Colab 기본 설치본 사용 (-U 금지: 실행 중 업그레이드하면 ImportError)

In [ ]:
import os, random, warnings
import numpy as np, pandas as pd
from scipy import sparse
from scipy.stats import rankdata
warnings.filterwarnings('ignore')
SEED=42; random.seed(SEED); np.random.seed(SEED)
RANGE='6m'          # '6m' 또는 'full'
DATA_DIR='/content'
need=[f'news_{RANGE}.parquet',f'reddit_{RANGE}.parquet',f'modelA_{RANGE}.csv']
if not all(os.path.exists(f'{DATA_DIR}/{f}') for f in need):
    from google.colab import files; files.upload()
news=pd.read_parquet(f'{DATA_DIR}/news_{RANGE}.parquet'); red=pd.read_parquet(f'{DATA_DIR}/reddit_{RANGE}.parquet')
A=pd.read_csv(f'{DATA_DIR}/modelA_{RANGE}.csv')
print(len(news),len(red),A.shape)

## 1. 임베딩

In [ ]:
# %%encoder
import torch
from sentence_transformers import SentenceTransformer
enc=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2'); enc.max_seq_length=128
def embed(texts): return enc.encode(list(texts),batch_size=512,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True).astype(np.float32)

In [ ]:
E_news=embed(news.text); E_red=embed(red.text)
np.save(f'E_news_{RANGE}.npy',E_news); np.save(f'E_red_{RANGE}.npy',E_red)   # 재실행 대비 캐시
DEFS={ # 정의문 3종: 표현 민감도 확인
 'a':("Positive news that is likely to raise the company's stock price.","Negative news that is likely to lower the company's stock price."),
 'b':("Strong earnings, revenue growth, new products, analyst upgrades, and optimistic outlook.","Weak earnings, revenue decline, lawsuits, analyst downgrades, and pessimistic outlook."),
 'c':("Investors are bullish and excited about this stock.","Investors are bearish and worried about this stock."),
}
DEFE={k:embed(v) for k,v in DEFS.items()}   # (2,d): pos, neg
print(E_news.shape,E_red.shape)

## 2. 일×종목 패널, 라벨, 평가 틀(PR #16과 동일)

In [ ]:
A=A.rename(columns={'date':'day'}); A['day']=A.day.astype(str)
panel=A[['day','ticker','real_ret','pred_ret','residual']].reset_index(drop=True)
days=np.array(sorted(panel.day.unique())); panel['di']=panel.day.map({d:i for i,d in enumerate(days)})
DI=panel.di.values; Y=panel.residual.values; REAL=panel.real_ret.values; PA=panel.pred_ret.values
day_rows=[np.flatnonzero(DI==i) for i in range(len(days))]
pos={k:i for i,k in enumerate((panel.ticker+'|'+panel.day).values)}

def agg(df,V):
    # 기사 단위 값 V(n,) 또는 (n,d) -> 패널 순서 일평균. 기사 없는 칸은 0, has=False
    idx=np.array([pos.get(k,-1) for k in (df.ticker+'|'+df.day).values]); ok=idx>=0
    V=np.asarray(V)[ok]; M=sparse.csr_matrix((np.ones(ok.sum()),(idx[ok],np.arange(ok.sum()))),shape=(len(panel),ok.sum()))
    cnt=np.asarray(M.sum(1)).ravel(); out=M@V; has=cnt>0
    out[has]=out[has]/(cnt[has].reshape(-1,*([1]*(out.ndim-1))))
    return out.astype(np.float32),has,cnt

def rankcorr(a,b):
    if len(a)<3 or np.ptp(a)==0 or np.ptp(b)==0: return np.nan
    return np.corrcoef(rankdata(a),rankdata(b))[0,1]
def daily_ic(pred,target,day_ids):
    return np.array([rankcorr(pred[day_rows[d]],target[day_rows[d]]) for d in day_ids])
def ic_summary(ic,horizon=3):
    ic=ic[~np.isnan(ic)]; se=ic.std(ddof=1)/np.sqrt(max(len(ic)/horizon,1.0))
    return ic.mean(), (ic.mean()/se if se>0 else np.nan)

PURGE=3; N=len(days); start=int(N*0.4); edges=np.linspace(start,N,6).astype(int)
FOLDS=[(np.arange(0,max(lo-PURGE,0)),np.arange(lo,hi)) for lo,hi in zip(edges[:-1],edges[1:])]
TEST_DAYS=np.concatenate([te for _,te in FOLDS])
rows_of=lambda ds: np.flatnonzero(np.isin(DI,ds))
print('days',N,'| fold test sizes',[len(te) for _,te in FOLDS])

def block_boot(x,block=5,n=2000,seed=0):
    x=np.asarray(x,float); x=x[~np.isnan(x)]; m=len(x); nb=int(np.ceil(m/block)); rng=np.random.default_rng(seed)
    st=rng.integers(0,m,(n,nb)); idx=((st[:,:,None]+np.arange(block))%m).reshape(n,-1)[:,:m]
    mu=x[idx].mean(1); return x.mean(),np.percentile(mu,2.5),np.percentile(mu,97.5)

## 3. 방법 정의 (모두 fold 안에서 학습 구간만 사용)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

def scalar_method(score_fn):
    # score_fn(tr_days)->(n_panel,) 점수. ε̂ = β·점수, β는 학습 구간 OLS(원점 통과). 기사 없는 칸은 0
    def f(tr,te,has):
        s=score_fn(tr); r=rows_of(tr); r=r[has[r]]
        beta=(s[r]@Y[r])/max(s[r]@s[r],1e-12); return np.where(has,beta*s,0.0)
    return f

def calibrated_model(emb,kind):
    grid={'ridge':[1e1,1e2,1e3,1e4,1e5],'mlp':[1e-1,1.0,10.0]}[kind]
    def make(h):
        if kind=='ridge': return Ridge(alpha=h)
        return MLPRegressor(hidden_layer_sizes=(32,16),alpha=h,learning_rate_init=1e-3,max_iter=200,early_stopping=True,
                            validation_fraction=0.15,n_iter_no_change=10,random_state=SEED)
    def f(tr,te,has):
        cut=int(len(tr)*0.8); fit_d,val_d=tr[:cut],tr[cut+PURGE:]
        rf=rows_of(fit_d); rf=rf[has[rf]]; rv=rows_of(val_d)
        ysd=Y[rows_of(tr)].std()+1e-12
        sc=StandardScaler().fit(emb[rows_of(tr)][has[rows_of(tr)]]); Z=sc.transform(emb)
        best=None
        for h in grid:
            p=np.where(has[rv],make(h).fit(Z[rf],Y[rf]/ysd).predict(Z[rv]),0.0) if len(rv) else np.zeros(0)
            ic=np.nanmean([rankcorr(p[np.isin(rv,day_rows[d])],Y[rv][np.isin(rv,day_rows[d])]) for d in val_d[PURGE*0:] if len(day_rows[d])]) if len(rv) else -9
            if best is None or ic>best[0]: best=(ic,h,p)
        ic,h,p=best; yv=Y[rv]
        beta=max((p@yv)/max(p@p,1e-12),0.0) if len(rv) else 0.0     # 검증 구간에서 크기 보정, 음수면 0
        ra=rows_of(tr); ra=ra[has[ra]]
        pred=make(h).fit(Z[ra],Y[ra]/ysd).predict(Z)
        return np.where(has,beta*pred,0.0)
    return f

def eps_centroid(emb,has):
    def score(tr):
        r=rows_of(tr); r=r[has[r]]; up=r[Y[r]>0]; dn=r[Y[r]<0]
        cp,cn=emb[up].mean(0),emb[dn].mean(0); cp/=np.linalg.norm(cp); cn/=np.linalg.norm(cn)
        return emb@cp-emb@cn
    return score

## 4. 실행: 소스별 특징 → 모든 방법 → 평가

In [ ]:
def run_method(f,has):
    eps=np.full(len(panel),np.nan)
    for tr,te in FOLDS:
        out=f(tr,te,has); r=rows_of(te); eps[r]=out[r]
    return eps

test_rows=rows_of(TEST_DAYS)
const=np.full(len(panel),np.nan)
for tr,te in FOLDS: const[rows_of(te)]=Y[rows_of(tr)].mean()
sse_c=((Y[test_rows]-const[test_rows])**2).sum()
ic_A=daily_ic(PA,REAL,TEST_DAYS)

def evaluate(name,source,eps):
    ic_e=daily_ic(eps,Y,TEST_DAYS); ic_f=daily_ic(PA+eps,REAL,TEST_DAYS)
    e,te_=ic_summary(ic_e); f,tf=ic_summary(ic_f); d,lo,hi=block_boot(ic_f-ic_A)
    sse=((Y[test_rows]-eps[test_rows])**2).sum()
    return dict(source=source,method=name,ic_eps=e,t_eps=te_,ic_final=f,t_final=tf,d_ic=d,d_lo=lo,d_hi=hi,r2_vs_const=1-sse/sse_c)

results=[]
def suite(source,df,E):
    d2=df[df.sent_score.notna()]; fb,has_fb,cnt=agg(d2,d2.sent_score.values); emb,has,_=agg(df,E)
    results.append(evaluate('FinBERT',source,run_method(scalar_method(lambda tr:fb),has_fb)))
    for k,(ep,en) in DEFE.items():
        zs,_,_=agg(df,E@ep-E@en)
        results.append(evaluate(f'정의유사도[{k}]',source,run_method(scalar_method(lambda tr,zs=zs:zs),has)))
    results.append(evaluate('ε-centroid',source,run_method(scalar_method(eps_centroid(emb,has)),has)))
    for kind in ['ridge','mlp']:
        results.append(evaluate(f'임베딩+{kind}',source,run_method(calibrated_model(emb,kind),has)))
    return emb,has

emb_n,has_n=suite('news',news,E_news)
emb_r,has_r=suite('reddit',red,E_red)
emb_b=np.hstack([emb_n,emb_r]); has_b=has_n|has_r
for kind in ['ridge','mlp']:
    results.append(evaluate(f'임베딩+{kind}','news+reddit',run_method(calibrated_model(emb_b,kind),has_b)))
# 참고 행
results.append(dict(source='-',method='(A only)',ic_final=ic_summary(ic_A)[0],t_final=ic_summary(ic_A)[1],d_ic=0.0))
results.append(dict(source='-',method='(const ε̄)',r2_vs_const=0.0))
R=pd.DataFrame(results); pd.set_option('display.width',200); pd.set_option('display.float_format','{:.4f}'.format)
display(R); R.to_csv(f'result_embed_v3_{RANGE}.csv',index=False)

## 5. 읽는 법
- `ic_eps`: ε̂과 실제 ε의 날짜별 IC. 진단용(ε와 A가 음의 상관이라 이것만으로는 착시 가능).
- `ic_final`/`d_ic`: **r̂=A+ε̂ vs real_ret**, d_ic는 A 단독 대비 차이와 블록 부트스트랩 95% 구간(`d_lo`,`d_hi`). 구간이 0을 포함하면 차이 없음.
- `r2_vs_const`: 학습 평균 ε만 쓴 기준선 대비 오차 감소(>0이어야 의미).
- 임베딩 방법이 FinBERT 행보다 나은지가 핵심이다. 시도한 방법 수(소스 3×방법 7 내외)가 많으므로 구간이 0을 살짝 벗어나는 정도는 우연일 수 있다.

## 6. 점검: 임베딩 점수가 감성 점수와 맞는가 / 입력 잘림

In [ ]:
from scipy.stats import spearmanr
for src,df,E in [('news',news,E_news),('reddit',red,E_red)]:
    z=E@DEFE['a'][0]-E@DEFE['a'][1]
    print(src,'기사 단위 Spearman(정의유사도[a], FinBERT) =',round(spearmanr(z[df.sent_score.notna().values],df.sent_score.dropna())[0],3))
    o=np.argsort(z); print('  하위3:',[df.text.iloc[i][:80] for i in o[:3]]); print('  상위3:',[df.text.iloc[i][:80] for i in o[-3:]])
tk=enc.tokenizer
for src,df in [('news',news),('reddit',red)]:
    s=df.text.sample(min(5000,len(df)),random_state=0).tolist()
    print(src,'128토큰 초과 비율 =',round(np.mean([len(tk(t)['input_ids'])>128 for t in s]),3))

In [ ]:
import matplotlib.pyplot as plt
P=R.dropna(subset=['d_lo']).copy(); P['label']=P.source+' / '+P.method
fig,ax=plt.subplots(figsize=(8,0.32*len(P)+1))
ax.errorbar(P.d_ic,range(len(P)),xerr=[P.d_ic-P.d_lo,P.d_hi-P.d_ic],fmt='o'); ax.axvline(0,color='gray')
ax.set_yticks(range(len(P))); ax.set_yticklabels(P.label); ax.set_xlabel('ΔIC (A+B − A 단독), 95% 블록 부트스트랩'); plt.tight_layout(); plt.savefig(f'delta_ic_{RANGE}.png',dpi=150); plt.show()
try:
    from google.colab import files; files.download(f'result_embed_v3_{RANGE}.csv'); files.download(f'delta_ic_{RANGE}.png')
except Exception: pass